### Phase 2: Feature Engineering (Der "Signal-Geber")
Modelle benötigen Input, um Regimes zu erkennen. Wir wandeln Rohdaten in stationäre Features um.

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
# --- Phase 2: Feature Engineering ---

import matplotlib.pyplot as plt
import pandas as pd

# Daten aus dem data-Ordner laden
df = pd.read_parquet(cfg.data_path("preprocessed"))

# Wir nutzen 'Cumulative_Returns' als unseren "Preis", da dies den Wert des 60/40 Portfolios über die Zeit darstellt

df['Vol_20'] = df['Returns'].rolling(cfg.features.volatility_window).std()
df['SMA_200'] = df['Cumulative_Returns'].rolling(cfg.features.sma_window).mean()
df['Distance_SMA'] = (df['Cumulative_Returns'] - df['SMA_200']) / df['SMA_200']
df['Momentum'] = df['Returns'].rolling(cfg.features.momentum_window).mean()
# Renditestrukturkurve (10Y - 3M Spread) - Ein inverser Spread (3M > 10Y) ist ein klassischer Rezessionsindikator
df['Yield_Spread'] = df['TNX_10Y'] - df['IRX_3M']

# Zeilen mit NaN-Werten (durch rolling) entfernen
df = df.dropna()

# Kurze Kontrolle
print("Features erfolgreich erstellt. Spalten im DataFrame:", df.columns.tolist())

# Visualisierung der Kapitalkurve des Portfolios
df['Cumulative_Returns'].plot(figsize=(12,6), title="Kapitalkurve des 60/40 Portfolios (Start = 1.0)")
# Capital Curve persistieren
plt.savefig(cfg.asset_path("capital_curve"), dpi=300, bbox_inches='tight')
plt.show()

print(df)

In [ ]:
# Korrelationsmatrix der Modell-Features
import seaborn as sns
import numpy as np

# Nur die 6 Modell-Features selektieren
feature_cols = cfg.features.model_features
corr_matrix = df[feature_cols].corr()

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # Obere Dreiecksmatrix ausblenden
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    ax=ax
)
ax.set_title("Pearson-Korrelationsmatrix der Modell-Features")
plt.tight_layout()

# Persistieren
corr_plot_path = cfg.asset_path("feature_correlation_matrix")
plt.savefig(corr_plot_path, dpi=300, bbox_inches='tight')
print(f"Korrelationsmatrix gespeichert unter: {corr_plot_path}")
plt.show()

# Numerische Matrix als Markdown speichern
corr_md_path = cfg.asset_path("feature_correlation_table")
corr_matrix.to_markdown(corr_md_path)

In [ ]:
output_path = cfg.data_path("feature_engineered")

# Speichern als Parquet
df.to_parquet(output_path)

print(f"Dataframe erfolgreich unter {output_path} gespeichert.")